# wavexplain — see why a forecast is what it is

Pick a product and read its **driver card**: how much of the forecast is a typical pattern, a promotion effect, and a recent trend. The three drivers plus the baseline sum **exactly** to the forecast, because each is a directly measured model prediction, not an estimated share.

This runs on public competition data (Corporacion Favorita). It describes the model's response to the data, not a real-world cause.


## Setup

In [ ]:
# --- Install ---
%pip -q install wavexplain torch numpy pandas matplotlib

# --- CONFIG: point these at your hosted checkpoint + a small sample panel ---
# The demo needs a trained checkpoint and a small slice of the panel the model
# was trained on (series the model has embeddings for). Host a few hundred
# series, not the full dataset.
CKPT_URL   = "https://github.com/kesjien/wavexplain/releases/download/demo/wavenet_demo.pt"   # TODO
DATA_URL   = "https://github.com/kesjien/wavexplain/releases/download/demo/sample_panel.npz"  # TODO
INPUT_LENGTH   = 90   # TODO: the window length your model expects
HORIZON        = 16    # TODO: match your trained model's horizon
NUM_SERIES     = 174685 # TODO: match training
NUM_COVARIATES = 1    # channel 1 = onpromotion

!wget -q -O wavenet_demo.pt "$CKPT_URL"   || echo "set CKPT_URL"
!wget -q -O sample_panel.npz "$DATA_URL"  || echo "set DATA_URL" 

In [ ]:
import numpy as np, torch, matplotlib.pyplot as plt
from collections import OrderedDict
from wavexplain import MultiSeriesWaveNet, CounterfactualExplainer, render_card_html
from IPython.display import HTML, display

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load the model (constructor args must match training)
model = MultiSeriesWaveNet(num_series=NUM_SERIES, horizon=HORIZON, num_covariates=NUM_COVARIATES)
model.load_state_dict(torch.load("wavenet_demo.pt", map_location=device))  # TODO: adapt if your ckpt is a dict
model.to(device).eval()

# Load a small sample panel: channel 0 = sales, channel 1 = onpromotion
z = np.load("sample_panel.npz")
sales_panel, promo_panel = z["sales_panel"], z["promo_panel"]   # (num_series, num_days)
num_days = sales_panel.shape[1]
start = num_days - INPUT_LENGTH - HORIZON
print("Loaded", sales_panel.shape[0], "series x", num_days, "days")

In [ ]:
def window(series_id):
    """Return the (log-sales, promo) input window for one series."""
    sales = sales_panel[series_id, start:start+INPUT_LENGTH].astype(float)
    promo = promo_panel[series_id, start:start+INPUT_LENGTH].astype(float)
    log_sales = np.log1p(np.clip(sales, 0, None))
    return np.stack([log_sales, promo]), sales, promo

def build_groups(promo, recent_k=14):
    """Split the input timesteps into typical / recent-trend / promotion groups."""
    T = len(promo)
    promo_mask  = promo > 0.5
    recent_mask = np.zeros(T, bool); recent_mask[-recent_k:] = True
    recent_mask &= ~promo_mask
    seasonal_mask = ~(promo_mask | recent_mask)
    return OrderedDict([
        ("seasonal_pattern", seasonal_mask),
        ("recent_trend",     recent_mask),
        ("promotion_effect", promo_mask),
    ])

def explain(series_id, full_input, promo):
    ex = CounterfactualExplainer(model, series_id=series_id, device=device, output_transform=torch.expm1)
    groups = build_groups(promo)
    contributions, baseline_pred, full_pred = ex.explain(
        full_input, baseline_values=[0.0]*(1+NUM_COVARIATES), reveal_groups=groups
    )
    return contributions, baseline_pred, full_pred

## Explain one product's forecast

In [ ]:
SERIES_ID = 256   # try any series id in the sample

full_input, sales, promo = window(SERIES_ID)
contributions, baseline_pred, full_pred = explain(SERIES_ID, full_input, promo)

print(f"Baseline:          {np.sum(baseline_pred):8.1f} units")
for name, val in contributions.items():
    print(f"  + {name:16s} {np.sum(val):+8.1f} units")
print(f"= Forecast (next {HORIZON}d): {np.sum(full_pred):8.1f} units")

# The plain-language card
render_card_html(
    title=f"Series {SERIES_ID}",
    total_forecast=full_pred,
    contributions=contributions,
    baseline_prediction=baseline_pred,
    highlight_group="promotion_effect",
    output_path="card.html",
)
display(HTML(open("card.html").read()))

## The same thing as a quick bar chart

In [ ]:
names  = ["baseline"] + list(contributions.keys())
values = [float(np.sum(baseline_pred))] + [float(np.sum(v)) for v in contributions.values()]

plt.figure(figsize=(8,4))
colors = ["tab:gray","tab:blue","tab:green","tab:orange"][:len(names)]
plt.bar(names, values, color=colors)
plt.axhline(0, color="k", lw=0.8)
plt.ylabel(f"Contribution to next-{HORIZON}d forecast (units)")
plt.title(f"Series {SERIES_ID}: what the forecast rests on")
plt.tight_layout(); plt.show()

## Notes

- Attribution is computed in the model's `log1p` space and converted back to units for display.
- Contributions plus baseline sum exactly to the forecast by construction.
- This explains series the model was trained on. Explaining a brand-new series requires fitting a model to it first.
- "Counterfactual" means the model's response to a controlled input change, not proven real-world causation.
